# DNN

In [1]:
import pandas as pd
import numpy as np
import re
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import StandardScaler
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
# ========== GLOBAL CONFIG ==========
CLUSTER = None
DATA_PATH = "/Users/xavierhua/Documents/GitHub/spotifynd/phase5_model_development"
PLAYLIST_FILE = f"{DATA_PATH}/playlist_final_final.csv"
TRACKS_FILE = f"{DATA_PATH}/tracks_new.csv"
NUM_PLAYLISTS = 1000
K_EVAL = 50

# Default Feature Weights
default_weights = {
    'popularity': 0.2,
    'era': 0.2,
    'length': 0.2,
    'sentiment': 0.2,
    'genre': 0.2
}

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Model Architecture
INPUT_SIZE = None  # will be determined later based on the dataset
HIDDEN_SIZES = [512, 256, 128]
DROPOUT_RATE = 0.2
ACTIVATION = nn.ReLU

# Training Parameters
BATCH_SIZE = 512
EPOCHS = 20
LEARNING_RATE = 0.001
WEIGHT_DECAY = 1e-4
LR_STEP_SIZE = 5
LR_GAMMA = 0.5
NEGATIVE_SAMPLE_RATIO = 10  # n_neg in PlaylistTrackDataset

# Loss multiplier
POS_WEIGHT_MULTIPLIER = 1.5

Using device: cpu


In [3]:
###################################
# 1. DATA LOADING & PREPROCESSING (DNN)
###################################
def load_data():
    playlists = pd.read_csv(PLAYLIST_FILE, engine='python', on_bad_lines='skip')
    tracks = pd.read_csv(TRACKS_FILE)
    if CLUSTER:
        playlists = playlists[playlists['cluster'] == CLUSTER].reset_index(drop=True)[:NUM_PLAYLISTS]
    else:
        playlists = playlists[:NUM_PLAYLISTS]
    return playlists, tracks

def split_data(playlists):
    """
    Splits playlists into train, validation, test, and final sets based on the 'dataset_type' column.
    """
    train = playlists[playlists['dataset_type'] == 'train'].reset_index(drop=True)
    val = playlists[playlists['dataset_type'] == 'val'].reset_index(drop=True)
    test = playlists[playlists['dataset_type'] == 'test'].reset_index(drop=True)
    final = playlists[playlists['dataset_type'] == 'final'].reset_index(drop=True)
    return train, val, test, final

def preprocess_playlist(playlist):
    # Convert centroids from strings to numpy arrays and unpack them
    playlist['sentiment_centroid'] = playlist['sentiment_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
    playlist['genre_centroid'] = playlist['genre_centroid'].apply(lambda x: np.fromstring(x.strip("[]"), sep=" "))
    
    sent_df = playlist['sentiment_centroid'].apply(pd.Series)
    sent_df.columns = [f'sent{i+1}' for i in range(sent_df.shape[1])]
    genre_df = playlist['genre_centroid'].apply(pd.Series)
    genre_df.columns = [f'genre{i+1}' for i in range(genre_df.shape[1])]
    
    playlist = pd.concat([playlist.drop(columns=['sentiment_centroid', 'genre_centroid']), sent_df, genre_df], axis=1)
    
    def convert_string_array_to_list(s):
        if isinstance(s, str):
            return [int(x) for x in re.findall(r'\d+', s)]
        return []
    
    playlist['track_idx_list'] = playlist['track_idx_list'].apply(convert_string_array_to_list)
    playlist['tracks_to_predict'] = playlist['tracks_to_predict'].apply(convert_string_array_to_list)
    
    cols = ['playlist_idx', 'dataset_type', 'track_idx_list', 'tracks_to_predict', 'cluster',
            'popularity_mean', 'era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion',
            'era_2000s_proportion', 'era_modern_era_proportion', 'length_short_proportion', 'length_medium_proportion',
            'length_long_proportion', 'sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6',
            'genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']
    return playlist[cols]

# Load and preprocess
playlist_raw, tracks = load_data()
playlist = preprocess_playlist(playlist_raw)
train_playlists_dnn, val_playlists_dnn, test_playlists_dnn, final_playlist_dnn = split_data(playlist)

# Define playlist feature columns and create scalers
playlist_popularity_cols = ['popularity_mean']
playlist_era_cols = ['era_early_years_proportion', 'era_classic_era_proportion', 'era_golden_era_proportion',
                       'era_2000s_proportion', 'era_modern_era_proportion']
playlist_length_cols = ['length_short_proportion', 'length_medium_proportion', 'length_long_proportion']
playlist_sentiment_cols = ['sent1', 'sent2', 'sent3', 'sent4', 'sent5', 'sent6']
playlist_genre_cols = ['genre1', 'genre2', 'genre3', 'genre4', 'genre5', 'genre6', 'genre7', 'genre8']

playlist_scalers = {
    'pop_playlist': StandardScaler(),
    'era_playlist': StandardScaler(),
    'len_playlist': StandardScaler(),
    'sent_playlist': StandardScaler(),
    'genre_playlist': StandardScaler()
}

playlist_scalers['pop_playlist'].fit(train_playlists_dnn[playlist_popularity_cols])
playlist_scalers['era_playlist'].fit(train_playlists_dnn[playlist_era_cols])
playlist_scalers['len_playlist'].fit(train_playlists_dnn[playlist_length_cols])
playlist_scalers['sent_playlist'].fit(train_playlists_dnn[playlist_sentiment_cols])
playlist_scalers['genre_playlist'].fit(train_playlists_dnn[playlist_genre_cols])

def normalize_playlist_features(df):
    df[playlist_popularity_cols] = playlist_scalers['pop_playlist'].transform(df[playlist_popularity_cols])
    df[playlist_era_cols] = playlist_scalers['era_playlist'].transform(df[playlist_era_cols])
    df[playlist_length_cols] = playlist_scalers['len_playlist'].transform(df[playlist_length_cols])
    df[playlist_sentiment_cols] = playlist_scalers['sent_playlist'].transform(df[playlist_sentiment_cols])
    df[playlist_genre_cols] = playlist_scalers['genre_playlist'].transform(df[playlist_genre_cols])
    return df

train_playlists_dnn = normalize_playlist_features(train_playlists_dnn.copy())
val_playlists_dnn = normalize_playlist_features(val_playlists_dnn.copy())
test_playlists_dnn = normalize_playlist_features(test_playlists_dnn.copy())

def create_playlist_feat_map(df):
    feat_cols = playlist_popularity_cols + playlist_era_cols + playlist_length_cols + playlist_sentiment_cols + playlist_genre_cols
    feat_map = {row['playlist_idx']: row[feat_cols].values for _, row in df.iterrows()}
    return feat_map

playlist_feat_map_train = create_playlist_feat_map(train_playlists_dnn)
playlist_feat_map_val = create_playlist_feat_map(val_playlists_dnn)
playlist_feat_map_test = create_playlist_feat_map(test_playlists_dnn)

# For tracks in DNN, define feature columns and scalers
popularity_cols = ['track_popularity']
era_cols = ['Early Years', 'Classic Era', 'Golden Era', '2000s', 'Modern Era']
length_cols = ['Short', 'Medium', 'Long']
sentiment_cols = ['joy', 'calm', 'sadness', 'fear', 'energizing', 'dreamy']
genre_cols = ['Instrumental / Ambient Sounds', 'Soft Acoustic / Classical', 'Orchestral / Soundtrack',
              'Mid-tempo Pop / Indie', 'Upbeat Electronic / Dance', 'Slow & Melancholic (Sad Songs)',
              'Experimental / Jazz Fusion', 'Lo-Fi / Chill Vibes']

track_scalers = {
    'pop_track': StandardScaler(),
    'era_track': StandardScaler(),
    'len_track': StandardScaler(),
    'sent_track': StandardScaler(),
    'genre_track': StandardScaler()
}

track_scalers['pop_track'].fit(tracks[popularity_cols])
track_scalers['era_track'].fit(tracks[era_cols])
track_scalers['len_track'].fit(tracks[length_cols])
track_scalers['sent_track'].fit(tracks[sentiment_cols])
track_scalers['genre_track'].fit(tracks[genre_cols])

def normalize_track_features(df):
    df[popularity_cols] = track_scalers['pop_track'].transform(df[popularity_cols])
    df[era_cols] = track_scalers['era_track'].transform(df[era_cols])
    df[length_cols] = track_scalers['len_track'].transform(df[length_cols])
    df[sentiment_cols] = track_scalers['sent_track'].transform(df[sentiment_cols])
    df[genre_cols] = track_scalers['genre_track'].transform(df[genre_cols])
    return df

tracks = normalize_track_features(tracks.copy())

def create_track_feat_map(df):
    feat_cols = popularity_cols + era_cols + length_cols + sentiment_cols + genre_cols
    feat_map = {int(row['track_idx']): row[feat_cols].values for _, row in df.iterrows()}
    return feat_map

track_feat_map = create_track_feat_map(tracks)

def apply_weights(features, weights):
    split_sizes = [1, 5, 3, 6, 8]
    chunks = np.split(features, np.cumsum(split_sizes)[:-1])
    return np.concatenate([chunk * weights[key] for chunk, key in zip(chunks, weights)])

In [4]:
###################################
# 2. MODEL & DATASET DEFINITION (DNN)
###################################
class PlaylistTrackDataset(Dataset):
    def __init__(self, playlist_df, feat_map, track_map, n_neg=NEGATIVE_SAMPLE_RATIO, cluster_weights=None):
        self.samples = []
        self.track_map = track_map
        all_tids = list(track_map.keys())
        cluster_weights = cluster_weights or {}
        for _, row in playlist_df.iterrows():
            pid, cluster = row['playlist_idx'], row['cluster']
            pos_tracks = row['tracks_to_predict']
            if not pos_tracks:  # skip if no positive tracks
                continue
            p_feat = feat_map[pid]
            weights = cluster_weights.get(cluster, default_weights)
            p_feat_w = apply_weights(p_feat, weights)
            for tid in pos_tracks:
                if tid in track_map:
                    self.samples.append((p_feat_w, track_map[tid], 1))
            negs = np.random.choice(list(set(all_tids) - set(pos_tracks)), 
                                    min(len(pos_tracks) * n_neg, len(all_tids)), replace=False)
            for tid in negs:
                self.samples.append((p_feat_w, track_map[tid], 0))
    def __len__(self):
        return len(self.samples)
    def __getitem__(self, idx):
        p, t, y = self.samples[idx]
        p = np.array(p, dtype=np.float32).flatten()
        t = np.array(t, dtype=np.float32).flatten()
        return torch.tensor(np.concatenate([p, t]), dtype=torch.float32), torch.tensor([y], dtype=torch.float32)

class DNNRecommender(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        layers = []
        prev_size = input_size
        for hidden in HIDDEN_SIZES:
            layers.append(nn.Linear(prev_size, hidden))
            layers.append(ACTIVATION())
            layers.append(nn.BatchNorm1d(hidden))
            layers.append(nn.Dropout(DROPOUT_RATE))
            prev_size = hidden
        layers.append(nn.Linear(prev_size, 1))
        self.model = nn.Sequential(*layers)
    def forward(self, x):
        return self.model(x).squeeze()

# TRAIN

In [5]:
# Create playlist feature maps for training and validation
playlist_feat_map_train = create_playlist_feat_map(train_playlists_dnn)
playlist_feat_map_val = create_playlist_feat_map(val_playlists_dnn)

# Create the dataset and DataLoader for training
train_dataset = PlaylistTrackDataset(train_playlists_dnn, playlist_feat_map_train, track_feat_map)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)

# Initialize model, optimizer, and scheduler
model = DNNRecommender(input_size=len(next(iter(train_dataset))[0])).to(device)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=LR_STEP_SIZE, gamma=LR_GAMMA)

# Compute pos_weight for loss function
pos = sum(1 for _, _, l in train_dataset.samples if l == 1)
neg = sum(1 for _, _, l in train_dataset.samples if l == 0)
pos_weight_val = (neg/pos * POS_WEIGHT_MULTIPLIER)
criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(pos_weight_val).to(device))

# Training loop
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        loss = criterion(model(xb).view(-1), yb.view(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    scheduler.step()
    print(f"Epoch {epoch+1}, Loss: {total_loss:.4f}")

Epoch 1, Loss: 187.9429
Epoch 2, Loss: 168.3156
Epoch 3, Loss: 160.4839
Epoch 4, Loss: 155.3731
Epoch 5, Loss: 151.5812
Epoch 6, Loss: 144.0152
Epoch 7, Loss: 141.6550
Epoch 8, Loss: 139.5506
Epoch 9, Loss: 136.8710
Epoch 10, Loss: 136.5851
Epoch 11, Loss: 131.8949
Epoch 12, Loss: 129.9426
Epoch 13, Loss: 128.2074
Epoch 14, Loss: 127.2542
Epoch 15, Loss: 126.0693
Epoch 16, Loss: 123.1966
Epoch 17, Loss: 122.7592
Epoch 18, Loss: 121.6132
Epoch 19, Loss: 120.9792
Epoch 20, Loss: 120.3581


# VALIDATE

In [6]:
def predict_dnn(playlist_idx, feat_map, track_feat_map):
    p_feat = feat_map[playlist_idx]
    p_feat_w = apply_weights(p_feat, default_weights)
    all_track_ids = list(track_feat_map.keys())
    p_feat_tensor = torch.tensor(np.array(p_feat_w, dtype=np.float32)).repeat(len(all_track_ids), 1).to(device)
    t_feats = np.array([track_feat_map[tid] for tid in all_track_ids], dtype=np.float32)
    t_tensor = torch.tensor(t_feats, dtype=torch.float32).to(device)
    inputs = torch.cat([p_feat_tensor, t_tensor], dim=1)
    model.eval()
    with torch.no_grad():
        scores = torch.sigmoid(model(inputs)).cpu().numpy().flatten()
    return scores

def get_all_dnn_predictions(feat_map, track_feat_map, test_playlists):
    predictions = {}
    for i, row in test_playlists.iterrows():
        playlist_id = row['playlist_idx']
        predictions[playlist_id] = predict_dnn(playlist_id, feat_map, track_feat_map)
    return predictions

def compute_metrics_for_playlist(predicted_tracks, true_tracks, k):
    top_k = predicted_tracks[:k]
    hit = int(any(t in top_k for t in true_tracks))
    precisions = []
    mrr = 0.0
    num_hits = 0
    for rank, track_id in enumerate(top_k):
        if track_id in true_tracks:
            num_hits += 1
            precisions.append(num_hits / (rank + 1))
            if mrr == 0.0:
                mrr = 1.0 / (rank + 1)
    ap = np.mean(precisions) if precisions else 0.0
    return hit, mrr, ap

def evaluate_model(model, test_playlists, playlist_feat_map, track_feat_map, k=K_EVAL, weights_by_cluster=None):
    model.eval()
    all_track_ids = list(track_feat_map.keys())
    total_hit, total_mrr, total_ap = 0, 0, 0
    n = len(test_playlists)
    with torch.no_grad():
        for _, row in test_playlists.iterrows():
            pid = row['playlist_idx']
            cluster = row['cluster']
            true_tracks = row['tracks_to_predict']
            seen_tracks = set(row['track_idx_list'])
            p_feat = playlist_feat_map[pid]
            weights = weights_by_cluster.get(cluster, default_weights) if weights_by_cluster else default_weights
            p_feat_w = apply_weights(p_feat, weights)
            p_feat_tensor = torch.tensor(np.array(p_feat_w, dtype=np.float32)).repeat(len(all_track_ids), 1)
            t_feats = np.array([track_feat_map[tid] for tid in all_track_ids], dtype=np.float32)
            t_tensor = torch.tensor(t_feats, dtype=torch.float32)
            inputs = torch.cat([p_feat_tensor, t_tensor], dim=1)
            scores = torch.sigmoid(model(inputs.to(device))).cpu().numpy().flatten()
            ranked_tracks = [tid for tid, _ in sorted(zip(all_track_ids, scores), key=lambda x: x[1], reverse=True)]
            ranked_unseen = [tid for tid in ranked_tracks if tid not in seen_tracks]
            hit, mrr, ap = compute_metrics_for_playlist(ranked_unseen, true_tracks, k)
            total_hit += hit
            total_mrr += mrr
            total_ap += ap
    return {
        'Hit@K': total_hit / n,
        'MRR': total_mrr / n,
        'MAP@K': total_ap / n
    }

# Use validation set for evaluation
playlist_feat_map_val = create_playlist_feat_map(val_playlists_dnn)
dnn_predictions_val = get_all_dnn_predictions(playlist_feat_map_val, track_feat_map, val_playlists_dnn)
val_metrics = evaluate_model(model, val_playlists_dnn, playlist_feat_map_val, track_feat_map, k=K_EVAL)
print("DNN Validation Evaluation:")
print(f"Hit@{K_EVAL}:             {val_metrics['Hit@K']:.4f}")
print(f"MRR:                    {val_metrics['MRR']:.4f}")
print(f"MAP@{K_EVAL}:             {val_metrics['MAP@K']:.4f}")

# Export recommendations from validation set (for demonstration)
val_recommendations = []
all_track_ids = list(track_feat_map.keys())
for _, row in val_playlists_dnn.iterrows():
    playlist_id = row['playlist_idx']
    scores = predict_dnn(playlist_id, playlist_feat_map_val, track_feat_map)
    ranked_indices = np.argsort(-scores)
    top_indices = ranked_indices[:K_EVAL]
    recommended_tracks = [all_track_ids[i] for i in top_indices if i < len(all_track_ids)]
    val_recommendations.append({'playlist_idx': playlist_id, 'recommended_tracks': recommended_tracks})
val_rec_df = pd.DataFrame(val_recommendations)
print(val_rec_df.head())

DNN Validation Evaluation:
Hit@50:             0.1327
MRR:                    0.0122
MAP@50:             0.0113
   playlist_idx                                 recommended_tracks
0             4  [111982, 140881, 15196, 110973, 95030, 176611,...
1             6  [188245, 9182, 71969, 641, 7416, 20529, 149917...
2            12  [9182, 71969, 188245, 191316, 149917, 239346, ...
3            49  [17801, 68607, 221989, 230872, 109597, 246267,...
4            66  [9182, 71969, 96735, 149917, 188245, 18002, 16...


# TEST

In [7]:
# Use test set for evaluation
playlist_feat_map_test = create_playlist_feat_map(test_playlists_dnn)
dnn_predictions_test = get_all_dnn_predictions(playlist_feat_map_test, track_feat_map, test_playlists_dnn)
test_metrics = evaluate_model(model, test_playlists_dnn, playlist_feat_map_test, track_feat_map, k=K_EVAL)
print("DNN Test Evaluation:")
print(f"Hit@{K_EVAL}:             {test_metrics['Hit@K']:.4f}")
print(f"MRR:                    {test_metrics['MRR']:.4f}")
print(f"MAP@{K_EVAL}:             {test_metrics['MAP@K']:.4f}")

# Export recommendations from test set
test_recommendations = []
all_track_ids = list(track_feat_map.keys())
for _, row in test_playlists_dnn.iterrows():
    playlist_id = row['playlist_idx']
    scores = predict_dnn(playlist_id, playlist_feat_map_test, track_feat_map)
    ranked_indices = np.argsort(-scores)
    top_indices = ranked_indices[:K_EVAL]
    recommended_tracks = [all_track_ids[i] for i in top_indices if i < len(all_track_ids)]
    test_recommendations.append({'playlist_idx': playlist_id, 'recommended_tracks': recommended_tracks})
test_rec_df = pd.DataFrame(test_recommendations)
print(test_rec_df.head())

DNN Test Evaluation:
Hit@50:             0.1731
MRR:                    0.0310
MAP@50:             0.0270
   playlist_idx                                 recommended_tracks
0             2  [145133, 42751, 7173, 197519, 224507, 242925, ...
1             7  [131348, 177774, 163078, 53478, 156665, 247320...
2            10  [157291, 191291, 3876, 53519, 84302, 74012, 16...
3            17  [191316, 9182, 71969, 189131, 51702, 188245, 1...
4            28  [149917, 71969, 163621, 18002, 9182, 95651, 13...
